In [1]:
ROLE = "Destination"
import paho.mqtt.client as mqtt
import time
import numpy as np
import matplotlib.pyplot as plt
import hmac
import pymongo
import copy

import sys
sys.path.append('../../')

import src

conf = src.CONFIG()
rx = src.RX(role=ROLE,conf=conf)

[INFO] [UHD] linux; GNU C++ version 13.3.0; Boost_108300; UHD_4.7.0.0-149-g635ad362
[INFO] [B200] Detected Device: B210
[INFO] [B200] Operating over USB 3.
[INFO] [B200] Initialize CODEC control...
[INFO] [B200] Initialize Radio control...
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Performing register loopback test... 
[INFO] [B200] Register loopback test passed
[INFO] [B200] Setting master clock rate selection to 'automatic'.
[INFO] [B200] Asking for clock rate 16.000000 MHz... 
[INFO] [B200] Actually got clock rate 16.000000 MHz.
[INFO] [B200] Asking for clock rate 32.000000 MHz... 
[INFO] [B200] Actually got clock rate 32.000000 MHz.
[INFO] [B200] Asking for clock rate 40.000000 MHz... 
[INFO] [B200] Actually got clock rate 40.000000 MHz.


In [ ]:
def run():
    global cnt_good, cnt_bad, collection
    src.MQTT_RX(conf=conf, role=ROLE).send_ready_and_wait_for_begin()
    file = rx.record()

    demod = src.rx.Demodulation(conf=conf)
    pp = src.PostProcessing(file=file, conf=conf, demod=demod, role=ROLE, plot=False)

    if pp.check():
        print("Recording is correct")
        for i in range(len(pp.Frames)):
            frame = pp.frameByNumber(i)
            hard_decision,rs, SNR = demod.decode(frame)
            index = demod.detect_message_indices(received=list(hard_decision), preamble=conf.PREAMBLE, repeat=conf.PREAMBLE_REPEAT)
            if index[0] is None or index[1] is None:
                print("No preamble detected!")
                continue

            msg_hard_decision = hard_decision[index[0]:index[1]]
            print("Message: ", pp.bits_to_string(msg_hard_decision[0:-256]))
            # print("recieved MAC: ", pp.binary_list_to_hex(msg_hard_decision[-256:]))
            expected_MAC = hmac.new(conf.MAC_KEY.encode('utf-8'), msg=pp.bits_to_string(msg_hard_decision[0:-256]).encode('utf-8'), digestmod='sha256').hexdigest()
            if pp.binary_list_to_hex(msg_hard_decision[-256:]) == expected_MAC:
                print("Good MAC")
                mac = True
                cnt_good += 1
            else:
                print("MAC is not correct")
                mac = False
                cnt_bad += 1

            SNR = SNR[index[0]+10:index[1]-10]
            print("SNR: ", np.nanmean(SNR))

            insert={
                'SNR': float(np.nanmean(SNR)),
                'MAC': pp.binary_list_to_hex(msg_hard_decision[-256:]),
                'message': pp.bits_to_string(msg_hard_decision[0:-256]),
                'integrity': mac,
                'time': time.time(),
                'config': copy.deepcopy(conf.config)
            }
            collection.insert_one(insert)

cnt_good = 0
cnt_bad = 0

myclient = pymongo.MongoClient("mongodb://localhost:27017/")
mydb = myclient["MAC_SUPERPOSITION"]
collection = mydb[f'1D_{ROLE}']
while True:
    print(fr"{cnt_good=}, {cnt_bad=}")
    run()

cnt_good=0, cnt_bad=0
[DESTINATION RX] Publishing 'ready'
[DESTINATION RX] Connected. Subscribing to 'mkashani/feeds/begin'
[DESTINATION RX] Received 'begin'
[DESTINATION RX] 'Begin' detected.

 Recorded Time: 4.069669485092163

number of frames: 10
# Frame OK ...

file size is: 159989760
Size check failed ...    
Recording is correct
Message:  Ôhis message is the default payload for the tests, and is 1088 bits long. It will be superposed with MAC tag of 256 bits with Rate= 1/3?This message is the defawlt payload for the tests, and is 1088 bits long. It will be superposed with MAC tag of 256 bits with Rate= 1/3?This message is phe default payload fos the tests, and is 1088 bits long. It will be superposed with MAC tag of 256 bits with Rate= 1/3?
MAC is not correct
SNR:  7.1074867


InvalidDocument: cannot encode object: <src.config.CONFIG object at 0x725253e333e0>, of type: <class 'src.config.CONFIG'>